# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 4: *Feature Engineering*
##### Version Number: 2.0
---
### Contents  
> 1. *Load Data*
> 2. *Days Without Rain*
> 3. *Fire History*
> 4. *Lagged Variables*
> 5. *Export Data*
---
### Notes
> Features Added:
> - `Days_without_Rain` A rolling count that serves as a simple drought indicator
> - `Fire_History` An average count of fires per month in a region spanning the alst two years (needs refinement)
> - `Lagged_Variables` 7 day rolling averages for 'Avg Air Temp (F)', 'Precip (in)', 'Avg Rel Hum (%)', 'Avg Wind Speed (mph)'
---
### Inputs
- `trimmed.csv` cleaned weather data joined with cleaned fire damage dataset
---
### Outputs 
- `final.csv` - final clean dataset with features added
---
### User Defined Dependencies

In [ ]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Function to print a grid of kde plots in consistent format, adjusts columns and rows accordingly
from src.plot_utils import grid_kde

# Function to print a custom format correlation heatmap
from src.plot_utils import correlation_map

# Function to calculate dryness index and return a dataframe
from src.data_utils import calculate_dryness_index

---
### Third Party Dependencies

In [ ]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats.mstats import winsorize


# Set consistent plotting style
sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

---

## 1. Load Datasets

In [4]:
# Load cleaned main dataset
final = pd.read_csv("../data/processed/trimmed.csv")

In [5]:
final.columns

Index(['Sample_ID', 'Date', 'Sample_Elevation', 'Region_ID', 'ETo (in)',
       'Precip (in)', 'Sol Rad (Ly/day)', 'Avg Vap Pres (mBars)',
       'Avg Air Temp (F)', 'Avg Rel Hum (%)', 'Avg Wind Speed (mph)',
       'Avg Soil Temp (F)', 'Season', 'Total Population', 'density',
       'Mean Income', 'Target'],
      dtype='object')

In [6]:
final

,Sample_ID,Date,Sample_Elevation,Region_ID,ETo (in),Precip (in),Sol Rad (Ly/day),Avg Vap Pres (mBars),Avg Air Temp (F),Avg Rel Hum (%),Avg Wind Speed (mph),Avg Soil Temp (F),Season,Total Population,density,Mean Income,Target
0,1,2018-01-01,36.609302,7,0.05,0.02,230.0,14.3,55.4,96.0,3.1,56.8,0,3269973,777.337917,111241,0
1,33,2018-01-01,1096.443481,5,0.10,0.00,260.0,4.1,51.9,31.0,4.1,53.8,0,441257,161.331803,111731,0
2,144,2018-01-01,1039.047485,1,0.06,0.00,228.0,8.4,48.9,71.0,3.4,47.0,0,64896,22.000807,74237,0
3,32,2018-01-01,288.821625,5,0.15,0.00,449.0,9.5,59.2,55.0,3.9,61.8,0,441257,161.331803,111731,0
4,145,2018-01-01,1615.276733,2,0.05,0.00,222.0,8.8,47.1,80.0,1.8,45.9,0,207172,126.597656,79233,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447775,52,2025-01-23,925.951294,6,0.10,0.00,347.0,1.5,43.7,16.0,4.9,40.6,0,2195611,109.468892,85327,0
447776,124,2025-01-23,473.016693,1,0.07,0.00,276.0,6.2,45.2,60.0,2.5,46.0,0,89108,25.413394,74102,0
447777,51,2025-01-23,1046.279663,6,0.10,0.00,347.0,1.5,43.7,16.0,4.9,40.6,0,2195611,109.468892,85327,0
447778,50,2025-01-23,956.628479,4,0.15,0.00,333.0,1.2,39.9,14.0,1.8,49.7,0,913820,112.374445,75161,0


---

## 2. Days Without Rain

flag to record whether **signifigant ( >1 inch)** precipitation has occured on a day

In [7]:
stations = final['Sample_ID'].unique()

final['Days Without Rain'] = 0
day_count = 0;

for station in stations:
    station_indexes = final[final['Sample_ID'] == station].index
    
    for index, row in final.loc[station_indexes].iterrows():
        if row['Precip (in)'] > 0:
            day_count = 0
        else:
            day_count = day_count + 1
    
        final.at[index, 'Days Without Rain'] = day_count


In [8]:
final['Days Without Rain']

0           0
1           8
2          31
3          21
4         268
         ... 
447775     10
447776      3
447777     10
447778    183
447779     17
Name: Days Without Rain, Length: 447780, dtype: int64

---

## 3.  Historical fires per month

Create a new column that records the historical average number of fires for each month, based on all previous years up to that point. Each row should reflect the average number of fires that occurred in the same calendar month across prior years, providing a historical baseline for comparison.

**ASSUMPTION** -Since the weather coverage in this dataset is limited, no previous year readings exists.
For now, I will just use the current count as a rough estimation for the average. 

In [9]:
final['Date'] = pd.to_datetime(final['Date'])

# Convert Datetime column to string
final['Month_Year'] = final['Date'].dt.strftime('%Y-%m')
month_year = final['Month_Year'].unique()

In [10]:
# Dictionary to store month-to-count mapping
monthly_fire_counts = {}

# Step 1: Calculate fire counts for each month
for month in month_year:
    # Get index and subset
    month_index = final[final['Month_Year'] == month].index
    subset = final.loc[month_index]
    
    # Eliminate 'Low' days
    total = subset[subset['Target'] != 0]
    
    # Store count
    monthly_fire_counts[month] = len(total)

# Step 2: Calculate 2-year rolling average
# Convert month_year to sorted list to ensure order
month_year_sorted = sorted(month_year)

# Dictionary for rolling averages
rolling_avg = {}

for i, month in enumerate(month_year_sorted):
    if i == 0:
        # First year: average of all *future* years (exclude current)
        future_values = list(monthly_fire_counts.values())[1:]
        rolling_avg[month] = np.mean(future_values) if future_values else 0

    elif i == 1:
        # Second year: average of first year and current
        prev = month_year_sorted[0]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev], monthly_fire_counts[month]])
    
    else:
        # Rolling average of previous 2 years
        prev1 = month_year_sorted[i - 1]
        prev2 = month_year_sorted[i - 2]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev1], monthly_fire_counts[prev2]])

# Step 3: Assign back to each row in the dataframe
for month in month_year_sorted:
    month_index = final[final['Month_Year'] == month].index
    final.loc[month_index, '2-Year Avg Fires'] = rolling_avg[month]


In [11]:
drop = ['Month_Year']

---

## 4. Add Lagged Weather Variables

We calculate 7-day rolling averages for select weather variables to capture recent trends that may influence fire severity.

In [14]:
# Sort data by County and Date to prepare for rolling average
final = final.sort_values(by=['Sample_ID', 'Date'])

# Define columns for 7-day rolling average
avg_columns = [
    'Avg Air Temp (F)', 
    'Precip (in)',
    'Avg Rel Hum (%)', 
    'Avg Wind Speed (mph)'
]

# Apply rolling mean by County
for col in avg_columns:
    final[f'{col} 7 Day Avg'] = final.groupby('Sample_ID')[col].rolling(window=7, min_periods=1).mean().reset_index(level=0, drop=True)

In [15]:
drop = ['Month_Year','PET_proxy']
final = final.drop(columns=drop)

---

## 5. Export Data

In [ ]:
final.to_csv("../data/processed/final.csv")
print("All datasets saved successfully to ../data/processed/")